# Ch.4 — Diffusion Models Today: Score SDE Framework

**Goal:** turn the discrete noise ladders from DDPM (Ch. 2) and NCSN (Ch. 3) into one
continuous-time framework, then train one time-dependent score model and sample it with
multiple numerical solvers.

**Structure**
1. From discrete noise levels to a continuous-time SDE
2. A unified `SDE` interface for VE-SDE (NCSN) and VP-SDE (DDPM)
3. Continuous-time denoising score matching
4. Reverse-time SDE sampling with Euler–Maruyama
5. Probability Flow ODE (PF-ODE) sampling with RK4
6. VP/DDPM-style ancestral sampling from the **same trained score model**
7. Compare stochastic vs. deterministic samplers + checkpoint

All experiments stay on normalized **2D two-moons data**, so the evolving marginals, score
field, and reverse trajectories remain visible. This follows the same "understand it in 2D first"
philosophy as the previous notebooks.


In [ ]:
# --- Setup (PROVIDED; keep the learning focus on the algorithms) --------
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# WORKFLOW: complete the TODOs in order, and run each SELF-CHECK cell.
# A NotImplementedError marks a deliberate exercise, not a notebook bug.
# Set True while debugging; use False for the full training/sampling experiment.
FAST_DEV_RUN = False

TRAIN_STEPS = 400 if FAST_DEV_RUN else 6000
SAMPLE_STEPS = 40 if FAST_DEV_RUN else 300
ODE_STEPS = 20 if FAST_DEV_RUN else 75
N_SAMPLES = 400 if FAST_DEV_RUN else 1200

# TODO (optional warm-up): print how many function evaluations (NFE) an RK4
# sampler uses with ODE_STEPS steps and compare that with SAMPLE_STEPS.


## 4.1 From Discrete Noise Levels to a Continuous-Time SDE

Both NCSN and DDPM can be written, over a tiny time increment $\Delta t$, in the common form
(Ch. 4, Eq. 4.1.1):

$$
x_{t+\Delta t}
\approx x_t + f(x_t,t)\Delta t + g(t)\sqrt{\Delta t}\,\epsilon_t,
\qquad \epsilon_t\sim\mathcal N(0,I).
$$

As $\Delta t\to 0$, this becomes the **forward SDE**

$$
dx(t) = f(x(t),t)\,dt + g(t)\,dw(t),
\qquad x(0)\sim p_{\text{data}}.
$$

The forward process defines a whole family of noisy marginals $p_t$. The object we learn is the
**time-dependent score**

$$
s(x,t) = \nabla_x \log p_t(x).
$$

Once that score is known, Ch. 4 gives two different generation dynamics with the **same marginal
densities**:

**Reverse-time SDE** (Eq. 4.1.6), solved from $T\to 0$:

$$
dx = \Big[f(x,t) - g(t)^2\nabla_x\log p_t(x)\Big]dt + g(t)\,d\bar w(t).
$$

**Probability Flow ODE** (Eq. 4.1.7):

$$
\frac{dx}{dt}
= f(x,t) - \frac12 g(t)^2\nabla_x\log p_t(x).
$$

That factor of $\tfrac12$ matters: the PF-ODE is **not** obtained by simply deleting the noise
term from the reverse SDE. Its drift is chosen so that its density evolution matches the same
$p_t$ family.


In [ ]:
# --- Toy dataset: normalized two moons (PROVIDED) -----------------------
X, _ = make_moons(n_samples=6000, noise=0.05, random_state=0)
X = torch.tensor(X, dtype=torch.float32)
X = (X - X.mean(0)) / X.std(0)  # normalize each coordinate

plt.figure(figsize=(4, 4))
plt.scatter(X[:, 0], X[:, 1], s=4, alpha=0.6)
plt.title("Target distribution: normalized two moons")
plt.axis("equal")
plt.show()

X_dev = X.to(device)

# TODO (optional warm-up): verify X.mean(0) ~ 0, X.std(0) ~ 1.
# TODO (optional warm-up): change the moons noise argument and visualize
# the effect on the data manifold. Restore noise=0.05 before training.


## 4.2 One Forward Kernel, Two Important SDEs

Section 4.5 gives a particularly clean way to think about the forward process. Instead of starting
from infinitesimal transitions, specify the accumulated perturbation kernel directly:

$$
p_t(x_t\mid x_0)
= \mathcal N(x_t;\alpha_t x_0,\sigma_t^2 I),
\qquad
x_t = \alpha_t x_0 + \sigma_t\epsilon,\quad \epsilon\sim\mathcal N(0,I).
$$

For this Gaussian kernel, the **conditional score** used by denoising score matching is closed-form:

$$
\nabla_{x_t}\log p_t(x_t\mid x_0)
= \frac{\alpha_t x_0-x_t}{\sigma_t^2}
= -\frac{\epsilon}{\sigma_t}.
$$

We implement both major Ch. 4 instantiations behind one interface:

| | **VE-SDE** | **VP-SDE** |
|---|---|---|
| Origin | continuous NCSN | continuous DDPM |
| Drift $f(x,t)$ | $0$ | $-\frac12\beta(t)x$ |
| Diffusion $g(t)$ | $\sqrt{d\,\sigma_{\text{sched}}^2(t)/dt}$ | $\sqrt{\beta(t)}$ |
| Signal coefficient $\alpha_t$ | $1$ | $\exp[-\frac12\int_0^t\beta(s)ds]$ |
| Prior | $\mathcal N(0,\sigma_{\max}^2I)$ (approx.) | $\mathcal N(0,I)$ |

For VE, the book's perturbation variance is
$\sigma_{\text{sched}}^2(t)-\sigma_{\text{sched}}^2(0)$, so the helper
`sigma(t)` below returns the **accumulated perturbation standard deviation**, which is zero at
$t=0$.


In [ ]:
# --- TODO 1: Unified SDE interface, VE-SDE, VP-SDE ---------------------
# Read §4.2 and the table in the preceding markdown. Write the math as
# vectorized PyTorch; NO Python loop over a batch of samples.
# Shapes: x0, x_t, noise = [B, 2]; t, alpha(t), sigma(t), g(t) = [B].
# Broadcast time-dependent coefficients to [B, 1] when multiplying x.
# Methods must work on CPU and CUDA without hard-coded tensor devices.

class SDE:
    T = 1.0
    eps = 1e-3  # do not evaluate the score at sigma(0) = 0

    def alpha(self, t):
        """Signal coefficient in x_t = alpha(t) * x0 + sigma(t) * eps."""
        raise NotImplementedError

    def sigma(self, t):
        """Accumulated perturbation std; NOT the instantaneous g(t)."""
        raise NotImplementedError

    def f_coeff(self, t):
        """Linear drift factor: f(x,t) = f_coeff(t) * x."""
        raise NotImplementedError

    def g(self, t):
        """Instantaneous diffusion coefficient (shape [B])."""
        raise NotImplementedError

    def drift(self, x, t):
        # TODO 1.1: Return f(x,t) using self.f_coeff(t).
        # HINT: x is [B,2], f_coeff(t) is [B].
        raise NotImplementedError("TODO 1.1: vectorized drift(x,t)")

    def q_sample(self, x0, t, noise=None):
        """Return (x_t, epsilon) where x_t|x0 is Gaussian."""
        # TODO 1.2: If noise is None, draw torch.randn_like(x0).
        # Compute x_t = alpha(t) * x0 + sigma(t) * noise.
        # Do NOT redraw noise if caller supplied it.
        # Return (x_t, noise) (both [B,2]).
        raise NotImplementedError("TODO 1.2: closed-form forward kernel")

    def conditional_score(self, noise, t):
        """Closed-form ∇_{x_t} log p_t(x_t | x0)."""
        # TODO 1.3: Use the relation between noise and sigma(t) above.
        # Output has the SAME shape as noise.
        raise NotImplementedError("TODO 1.3: conditional Gaussian score")

    def prior_sample(self, n, dim=2, generator=None):
        raise NotImplementedError


class VESDE(SDE):
    """Variance Exploding SDE; continuous-time analogue of NCSN.

    Schedule: sigma_sched(t) = sigma_min*(sigma_max/sigma_min)^t.
    Crucial distinction: sigma(t)^2 = sigma_sched(t)^2 - sigma_min^2,
    whereas g(t)^2 = d/dt [sigma_sched(t)^2].
    """
    def __init__(self, sigma_min=0.01, sigma_max=5.0):
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
        self.log_ratio = math.log(sigma_max / sigma_min)

    def sigma_sched(self, t):
        # TODO 1.4: Implement the geometric sigma schedule using
        # self.sigma_min and self.log_ratio. Return shape [B].
        raise NotImplementedError("TODO 1.4: VE geometric schedule")

    def alpha(self, t):
        # TODO 1.5: VE preserves the data signal (alpha_t = 1).
        # HINT: torch.ones_like(t).
        raise NotImplementedError("TODO 1.5: VE signal coefficient")

    def sigma(self, t):
        # TODO 1.6: Accumulated perturbation standard deviation, not
        # sigma_sched(t) itself. Square-root of the nonnegative variance.
        # Clamp before sqrt to protect against floating-point cancellation.
        raise NotImplementedError("TODO 1.6: VE accumulated sigma")

    def f_coeff(self, t):
        # TODO 1.7: What is the VE drift? Return a tensor of shape [B].
        raise NotImplementedError("TODO 1.7: VE drift coefficient")

    def g(self, t):
        # TODO 1.8: Derive g(t) from g(t)^2 = d[sigma_sched(t)^2]/dt.
        # Differentiate analytically; do not use autograd or finite differences.
        raise NotImplementedError("TODO 1.8: VE diffusion")

    def prior_sample(self, n, dim=2, generator=None):
        # TODO 1.9: Approximate p_T with N(0, sigma_max^2 I).
        # Create a [n,dim] tensor on `device`; honor `generator` for seeding.
        raise NotImplementedError("TODO 1.9: VE Gaussian prior")


class VPSDE(SDE):
    """Variance Preserving SDE; continuous-time analogue of DDPM.

    beta(t) = beta_min + t*(beta_max-beta_min)
    f(x,t) = -0.5*beta(t)*x, g(t) = sqrt(beta(t)).
    """
    def __init__(self, beta_min=0.1, beta_max=20.0):
        self.beta_min = beta_min
        self.beta_max = beta_max

    def beta(self, t):
        # TODO 1.10: Linear beta(t) schedule.
        raise NotImplementedError("TODO 1.10: beta(t)")

    def integral_beta(self, t):
        # TODO 1.11: Analytically integrate beta(u) from u=0 to u=t.
        # CHECK: ∫[a+bu]du = a*t + (b/2)*t^2.
        raise NotImplementedError("TODO 1.11: integral of beta")

    def alpha(self, t):
        # TODO 1.12: alpha_t = exp[-0.5 * integral_beta(t)].
        raise NotImplementedError("TODO 1.12: VP alpha")

    def sigma(self, t):
        # TODO 1.13: sigma_t^2 = 1 - exp[-integral_beta(t)].
        # Take a numerically safe square root.
        raise NotImplementedError("TODO 1.13: VP sigma")

    def f_coeff(self, t):
        # TODO 1.14: Coefficient of x in the VP forward drift.
        raise NotImplementedError("TODO 1.14: VP drift coefficient")

    def g(self, t):
        # TODO 1.15: Instantaneous diffusion coefficient (NOT sigma_t).
        raise NotImplementedError("TODO 1.15: VP diffusion")

    def prior_sample(self, n, dim=2, generator=None):
        # TODO 1.16: VP prior is standard normal; support `generator`.
        raise NotImplementedError("TODO 1.16: VP Gaussian prior")


# Construct the two configurations after completing TODO 1.1–1.16.
ve = VESDE()
vp = VPSDE()

print("VE terminal alpha, sigma:",
      ve.alpha(torch.tensor([1.0])).item(),
      ve.sigma(torch.tensor([1.0])).item())
print("VP terminal alpha, sigma:",
      vp.alpha(torch.tensor([1.0])).item(),
      vp.sigma(torch.tensor([1.0])).item())


In [ ]:
# --- SELF-CHECK 1: SDE shapes, marginal identities, seeded sampling ------
# Run only after completing TODO 1.*. These tests deliberately do not
# reveal implementation code; they detect common algebra/broadcast bugs.
t_test = torch.tensor([0.2, 0.7], device=device)
x_test = torch.tensor([[1.0, -2.0], [0.5, 1.0]], device=device)
fixed_noise = torch.tensor([[0.1, -0.4], [-1.0, 0.6]], device=device)

for name, sde in [("VE", ve), ("VP", vp)]:
    for fn in (sde.alpha, sde.sigma, sde.f_coeff, sde.g):
        y = fn(t_test)
        assert y.shape == t_test.shape, (name, fn.__name__, y.shape)
        assert torch.isfinite(y).all(), (name, fn.__name__)
    assert sde.drift(x_test, t_test).shape == x_test.shape
    xt, returned_noise = sde.q_sample(x_test, t_test, fixed_noise)
    expected_xt = (sde.alpha(t_test)[:, None] * x_test
                   + sde.sigma(t_test)[:, None] * fixed_noise)
    torch.testing.assert_close(xt, expected_xt, rtol=1e-5, atol=1e-6)
    torch.testing.assert_close(returned_noise, fixed_noise)
    torch.testing.assert_close(
        sde.conditional_score(fixed_noise, t_test),
        -fixed_noise / sde.sigma(t_test)[:, None],
        rtol=1e-5, atol=1e-6,
    )
    g = torch.Generator(device=device).manual_seed(123)
    prior = sde.prior_sample(128, generator=g)
    assert prior.shape == (128, 2) and torch.isfinite(prior).all()

# VP preserves unit variance for a standard-normal data distribution.
t_grid = torch.linspace(vp.eps, 1.0, 25, device=device)
torch.testing.assert_close(
    vp.alpha(t_grid)**2 + vp.sigma(t_grid)**2,
    torch.ones_like(t_grid), rtol=1e-5, atol=1e-5,
)
# VE drift is zero; its g^2 is the time derivative of accumulated variance.
torch.testing.assert_close(ve.f_coeff(t_grid), torch.zeros_like(t_grid))
delta = 5e-4
slope = (ve.sigma(t_grid + delta)**2 - ve.sigma(t_grid - delta)**2)/(2*delta)
assert torch.allclose(ve.g(t_grid)**2, slope, rtol=1e-2, atol=1e-2)
print("✓ SELF-CHECK 1 passed: VE/VP contracts and marginal identities")


In [ ]:
# --- Visualize forward marginals ----------------------------------------
@torch.no_grad()
def plot_forward_marginals(sde, title, times=(0.0, 0.25, 0.5, 0.75, 1.0), n=1500):
    fig, axes = plt.subplots(1, len(times), figsize=(3 * len(times), 3))

    x0 = X_dev[:n]
    shared_noise = torch.randn_like(x0)

    for ax, t_value in zip(axes, times):
        if t_value == 0.0:
            xt = x0
        else:
            t = torch.full((n,), t_value, device=device)
            xt, _ = sde.q_sample(x0, t, shared_noise)

        xt = xt.cpu()
        ax.scatter(xt[:, 0], xt[:, 1], s=3, alpha=0.5)
        ax.set_title(f"t={t_value:.2f}")
        ax.axis("equal")
        ax.set_xticks([])
        ax.set_yticks([])

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_forward_marginals(ve, "VE-SDE forward marginals: variance explodes")
plot_forward_marginals(vp, "VP-SDE forward marginals: signal shrinks, total variance stays bounded")

# TODO (optional plotting): Add a separate figure of alpha(t) and sigma(t)
# for VE vs. VP on the same t grid; explain their different terminal scales.


**What to notice**

- **VE-SDE:** the data are not contracted ($\alpha_t=1$); the cloud simply becomes broader and
  broader until the injected noise dominates the original geometry.
- **VP-SDE:** the signal coefficient $\alpha_t$ decays while the perturbation variance grows toward
  one. The process ends near a standard Gaussian.
- In both cases, training does **not** need to numerically simulate the forward SDE. Because the
  perturbation kernel is Gaussian, we can jump directly from $x_0$ to any randomly sampled $t$.


## 4.3 Continuous-Time Denoising Score Matching

The score $\nabla_x\log p_t(x)$ is unknown because $p_t$ is a marginal over all clean data points.
Exactly as in Ch. 3, conditioning on the known training example $x_0$ turns the target into a
closed-form Gaussian score.

Ch. 4, Eq. 4.3.1:

$$
\mathcal L_{\text{DSM}}(\phi)
=
\frac12\,
\mathbb E_{t,x_0,x_t}
\left[
\omega(t)
\left\|
s_\phi(x_t,t)
-
\nabla_{x_t}\log p_t(x_t\mid x_0)
\right\|_2^2
\right].
$$

For our affine Gaussian kernel,

$$
\nabla_{x_t}\log p_t(x_t\mid x_0)=-\epsilon/\sigma_t.
$$

The minimizer is the *marginal* score
$s^*(x_t,t)=\nabla_{x_t}\log p_t(x_t)$ (Proposition 4.3.1).

We use $\omega(t)=\sigma_t^2$. This is the same scale-balancing idea used in the NCSN notebook:
the target magnitude grows like $1/\sigma_t$, so multiplying squared error by $\sigma_t^2$ keeps
different times on a more comparable scale.


In [ ]:
# --- TODO 2: time-conditioned score network -----------------------------
# Write an MLP that takes both noisy coordinates and continuous time.
# Target I/O: x=[B,2], t=[B] -> predicted score=[B,2].

class SinusoidalTimeEmbed(nn.Module):
    """Continuous time embedding inspired by Ch. 2's DDPM embedding."""
    def __init__(self, dim=32):
        super().__init__()
        assert dim % 2 == 0, "Need equal numbers of sine/cosine features"
        self.dim = dim

    def forward(self, t):
        # TODO 2.1: Make half=dim//2 log-spaced frequencies:
        #   exp(-log(10000)*arange(half)/half)
        # Place arange on t.device. Form phase [B,half] by multiplying
        # (1000*t)[:,None] and frequencies[None,:].
        # Concatenate sin(phase), cos(phase) -> [B,dim].
        # No Python loops or NumPy here: use vectorized torch operations.
        raise NotImplementedError("TODO 2.1: sinusoidal embedding")


class TimeScoreNet(nn.Module):
    """s_phi(x_t,t): R^2 x [0,1] -> R^2."""
    def __init__(self, dim=2, hidden=128, t_dim=32):
        super().__init__()
        self.temb = SinusoidalTimeEmbed(t_dim)
        # TODO 2.2: Define self.net as an nn.Sequential MLP.
        # Input width = dim+t_dim; output width = dim.
        # Suggested: 3 hidden Linear layers, SiLU after each,
        # then an output Linear layer (no final activation).
        raise NotImplementedError("TODO 2.2: MLP in __init__")

    def forward(self, x, t):
        # TODO 2.3: Obtain the time embedding, concatenate it with x
        # along the FEATURES dimension, then pass it through self.net.
        # Assert mentally: each batch sample gets its own time embedding.
        raise NotImplementedError("TODO 2.3: concatenate x and time")


In [ ]:
# --- SELF-CHECK 2: embedding, MLP output shape, trainable gradients ------
t = torch.tensor([0.01, 0.3, 0.85, 1.0], device=device)
x = torch.randn(4, 2, device=device)
embed = SinusoidalTimeEmbed(32).to(device)
emb = embed(t)
assert emb.shape == (4, 32) and torch.isfinite(emb).all()
assert not torch.allclose(emb[0], emb[-1]), "Time embedding ignores time"

score_test = TimeScoreNet().to(device)
out_test = score_test(x, t)
assert out_test.shape == x.shape and torch.isfinite(out_test).all()
out_test.square().mean().backward()
assert any(p.grad is not None and torch.isfinite(p.grad).all()
           for p in score_test.parameters()), "No gradient to network parameters"
print("✓ SELF-CHECK 2 passed: time network and gradients")


In [ ]:
# --- TODO 3: Generic continuous-time DSM training loop -----------------
# Requirements: same loop must work unchanged for VE and VP.
# Minimize E_t,x,eps [sigma(t)^2 * ||s_phi(x_t,t)+eps/sigma(t)||^2].
# Sum squared error across the 2 coordinates, then average over batch.

def train_score_sde(model, sde, data, n_steps=8000, batch_size=256, lr=2e-3):
    # TODO 3.1: Instantiate torch.optim.Adam on model.parameters().
    raise NotImplementedError("TODO 3.1: initialize optimizer")

    for step in range(n_steps):
        # TODO 3.2: Choose `batch_size` random integer indices from data,
        # then gather x0 [B,2] on the same device as data.
        raise NotImplementedError("TODO 3.2: random minibatch")

        # TODO 3.3: Sample *independent per-example* uniform times in
        # [sde.eps, sde.T]. Avoid exactly t=0 (sigma would vanish).
        raise NotImplementedError("TODO 3.3: continuous random times")

        # TODO 3.4: Draw x_t and epsilon via sde.q_sample; derive the
        # known conditional-score target; run the model on (x_t,t).
        raise NotImplementedError("TODO 3.4: DSM inputs and target")

        # TODO 3.5: Compute omega(t)=sigma(t)^2 (shape [B]); use weighted
        # squared L2 error, summing over dim=1 and averaging the batch.
        # Factor 1/2 in the book is optional for this optimizer.
        raise NotImplementedError("TODO 3.5: correctly reduced DSM loss")

        # TODO 3.6: Standard three-step PyTorch optimization: zero grad,
        # backprop, parameter update. Avoid retaining old gradients.
        raise NotImplementedError("TODO 3.6: gradient descent step")

        if step % max(1, n_steps // 8) == 0:
            print(f"step {step:5d} | loss {loss.item():.4f}")

    return model


In [ ]:
# --- SELF-CHECK 3: training loop changes weights and returns model -------
# This is a 3-step plumbing check, not a test of convergence/quality.
small_model = TimeScoreNet(hidden=32).to(device)
before = [p.detach().clone() for p in small_model.parameters()]
returned = train_score_sde(small_model, vp, X_dev, n_steps=3, batch_size=16)
assert returned is small_model
assert any(not torch.equal(a, b) for a, b in zip(before, small_model.parameters()))
assert all(torch.isfinite(p).all() for p in small_model.parameters())
print("✓ SELF-CHECK 3 passed: model weights updated")


In [ ]:
# --- TODO 4A: train the VE score model (continuous NCSN) ----------------
torch.manual_seed(1)
# TODO 4.1: Instantiate a fresh TimeScoreNet, move it to `device`, then
# call train_score_sde on (that network, ve, X_dev, TRAIN_STEPS).
# Store the trained network as `score_ve` for later visualization/sampling.
raise NotImplementedError("TODO 4.1: create and train score_ve")


In [ ]:
# --- TODO 4B: train the VP score model (continuous DDPM) ----------------
torch.manual_seed(2)
# TODO 4.2: Train a *separate* TimeScoreNet using vp, not ve, with the
# same `train_score_sde` function. Store it as `score_vp`.
raise NotImplementedError("TODO 4.2: create and train score_vp")


In [ ]:
# --- Visualize the learned time-dependent score fields ------------------
@torch.no_grad()
def plot_score_field(model, t_value, ax, xlim=(-2.5, 2.5), ylim=(-2.5, 2.5), grid=18):
    xs = torch.linspace(*xlim, grid, device=device)
    ys = torch.linspace(*ylim, grid, device=device)
    xx, yy = torch.meshgrid(xs, ys, indexing="xy")
    pts = torch.stack([xx.flatten(), yy.flatten()], dim=1)
    t = torch.full((pts.shape[0],), t_value, device=device)

    score = model(pts, t).cpu()
    pts = pts.cpu()

    ax.quiver(pts[:, 0], pts[:, 1], score[:, 0], score[:, 1], alpha=0.55)
    ax.set_title(f"t={t_value:.2f}")
    ax.axis("equal")


fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, t_value in zip(axes[0], [0.05, 0.4, 0.9]):
    plot_score_field(score_ve, t_value, ax)
axes[0, 0].set_ylabel("VE score field")

for ax, t_value in zip(axes[1], [0.05, 0.4, 0.9]):
    plot_score_field(score_vp, t_value, ax)
axes[1, 0].set_ylabel("VP score field")

plt.tight_layout()
plt.show()

# TODO (optional plotting): implement your own function to visualize the
# learned score quiver field at a user-selected time t=0.1, 0.5, or 0.9.
# Compare how the length and smoothness of the vectors change with t.


At small $t$, the score field has to resolve the sharp local geometry of the two moons. At large
$t$, the noisy marginal is smoother and the field becomes much more global. This is the
continuous-time version of the coarse-to-fine score fields from NCSN — except now the network is
trained over a **continuum** of noise levels rather than a finite ladder.


## 4.4 Reverse-Time SDE Sampling — Euler–Maruyama

After training, replace the oracle score in the reverse SDE by $s_{\phi^\times}(x,t)$:

$$
dx
=
\left[
f(x,t)-g(t)^2s_{\phi^\times}(x,t)
\right]dt
+
g(t)d\bar w(t).
$$

Ch. 4, Eq. 4.3.4 applies **Euler–Maruyama** while integrating backward in time.
If our grid has $t_{\text{next}}<t$, define $h=t_{\text{next}}-t<0$:

$$
x_{\text{next}}
=
x
+
\left[f(x,t)-g(t)^2s_\phi(x,t)\right]h
+
g(t)\sqrt{-h}\,\epsilon.
$$

The drift uses the score to pull samples toward the evolving high-density region; the Brownian term
keeps the reverse process stochastic.


In [ ]:
# --- TODO 5: Reverse-time SDE sampler (Euler-Maruyama) ------------------
# Read §4.4. Reuse this sampler for BOTH VE and VP without branching.
# Important: t_next < t_now, so h < 0. Brownian std scales as sqrt(-h).
# A fixed `initial` and seed should make this sampler reproducible.

@torch.no_grad()
def reverse_sde_sample(
    model,
    sde,
    n_samples=2000,
    dim=2,
    n_steps=500,
    initial=None,
    seed=0,
    record_every=None,
):
    generator = torch.Generator(device=device).manual_seed(seed)

    # PROVIDED: use either user-supplied starting points or the SDE prior.
    if initial is None:
        x = sde.prior_sample(n_samples, dim, generator=generator)
    else:
        x = initial.to(device).clone()
        n_samples = x.shape[0]

    times = torch.linspace(sde.T, sde.eps, n_steps + 1, device=device)
    traj = [x.cpu().clone()] if record_every else None

    for i in range(n_steps):
        t_now = times[i]
        t_next = times[i + 1]
        h = t_next - t_now  # intentionally NEGATIVE
        t = torch.full((n_samples,), float(t_now), device=device)

        # TODO 5.1: Evaluate model(x,t), sde.g(t), sde.drift(x,t).
        # g(t) should broadcast as [B,1].
        raise NotImplementedError("TODO 5.1: score and SDE coefficients")

        # TODO 5.2: Implement reverse drift f(x,t)-g(t)^2*s_phi(x,t).
        # Draw independent N(0,I) increments with the provided generator.
        raise NotImplementedError("TODO 5.2: reverse drift and noise")

        # TODO 5.3: Take the Euler-Maruyama step:
        # x_next = x + reverse_drift*h + g(t)*sqrt(-h)*noise.
        # CAUTION: sqrt(h) is invalid here because h is negative.
        raise NotImplementedError("TODO 5.3: Euler-Maruyama update")

        if record_every and (i + 1) % record_every == 0:
            traj.append(x.cpu().clone())

    return x.cpu(), traj


In [ ]:
# --- SELF-CHECK 4: reverse sampler against an analytically known score --
# If data are N(0,I), the VP forward process is stationary N(0,I).
# Its exact marginal score is s(x,t)=-x, at ALL t. No trained network needed.
class StandardGaussianScore(nn.Module):
    def forward(self, x, t):
        return -x

oracle_gaussian = StandardGaussianScore().to(device)
start = torch.randn(96, 2, device=device)
a, traj = reverse_sde_sample(
    oracle_gaussian, vp, n_steps=32, initial=start, seed=123, record_every=8
)
b, _ = reverse_sde_sample(
    oracle_gaussian, vp, n_steps=32, initial=start, seed=123, record_every=8
)
assert a.shape == (96, 2) and len(traj) == 5  # start + 4 snapshots
assert torch.isfinite(a).all()
torch.testing.assert_close(a, b)  # same noise seed -> same path
print("✓ SELF-CHECK 4 passed: reverse SDE shape / snapshots / reproducibility")


In [ ]:
# --- Sample VE and VP with the same generic reverse-SDE solver ----------
samples_ve_sde, traj_ve = reverse_sde_sample(
    score_ve, ve,
    n_samples=N_SAMPLES,
    n_steps=SAMPLE_STEPS,
    seed=10,
    record_every=max(1, SAMPLE_STEPS // 5),
)

samples_vp_sde, traj_vp = reverse_sde_sample(
    score_vp, vp,
    n_samples=N_SAMPLES,
    n_steps=SAMPLE_STEPS,
    seed=11,
    record_every=max(1, SAMPLE_STEPS // 5),
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].scatter(X[:, 0], X[:, 1], s=3, alpha=0.5)
axes[0].set_title("Data")
axes[1].scatter(samples_ve_sde[:, 0], samples_ve_sde[:, 1], s=3, alpha=0.5)
axes[1].set_title("VE reverse SDE")
axes[2].scatter(samples_vp_sde[:, 0], samples_vp_sde[:, 1], s=3, alpha=0.5)
axes[2].set_title("VP reverse SDE")
for ax in axes:
    ax.axis("equal")
plt.tight_layout()
plt.show()

# TODO (optional experiment): keep model weights fixed, reduce SAMPLE_STEPS
# to 40 and inspect discretization artifacts in VE vs. VP samples.


In [ ]:
# --- Watch the VP reverse SDE turn noise into data ----------------------
fig, axes = plt.subplots(1, len(traj_vp), figsize=(3 * len(traj_vp), 3))
for ax, snap in zip(axes, traj_vp):
    ax.scatter(snap[:, 0], snap[:, 1], s=2, alpha=0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.axis("equal")
plt.suptitle("VP reverse SDE: prior -> data")
plt.tight_layout()
plt.show()

# TODO (optional plotting): give all frames identical x/y axis limits
# so the visible change in scale is not hidden by autoscaling.


## 4.5 Probability Flow ODE — Same Marginals, Deterministic Paths

The PF-ODE associated with the same forward SDE is

$$
\frac{dx}{dt}
=
f(x,t)
-
\frac12 g(t)^2 s_{\phi^\times}(x,t).
$$

Unlike the reverse SDE, this dynamics has **no Brownian term**, so a fixed initial latent produces
a fixed trajectory. Yet the Fokker–Planck / continuity-equation argument in Section 4.2 shows that
the PF-ODE is constructed to share the same marginal family $p_t$ as the forward/reverse SDEs.

Euler works, but an ODE lets us use standard higher-order solvers. Below we implement
**RK4 from scratch** to avoid an extra dependency.


In [ ]:
# --- TODO 6: PF-ODE vector field + Runge-Kutta 4 sampler ----------------
# Key distinction: the ODE has -0.5*g(t)^2 * score, NOT -g(t)^2 * score.
# No Brownian noise anywhere inside the integration loop.

@torch.no_grad()
def probability_flow_drift(model, sde, x, t_value):
    # TODO 6.1: Build a batch of times [B] from scalar t_value on x.device.
    # Then compute f(x,t) - 0.5*g(t)^2*s_phi(x,t) as a [B,2] tensor.
    raise NotImplementedError("TODO 6.1: probability-flow vector field")


@torch.no_grad()
def pf_ode_sample(
    model,
    sde,
    n_samples=2000,
    dim=2,
    n_steps=125,
    initial=None,
    seed=0,
    record_every=None,
):
    # PROVIDED: same initialization contract as the SDE sampler.
    generator = torch.Generator(device=device).manual_seed(seed)
    if initial is None:
        x = sde.prior_sample(n_samples, dim, generator=generator)
    else:
        x = initial.to(device).clone()
        n_samples = x.shape[0]

    times = torch.linspace(sde.T, sde.eps, n_steps + 1, device=device)
    traj = [x.cpu().clone()] if record_every else None

    for i in range(n_steps):
        t = times[i]
        t_next = times[i + 1]
        h = t_next - t  # negative when solving backwards

        # TODO 6.2: Implement one classical RK4 step with vector field F:
        # k1=F(x,t), k2=F(x+h*k1/2,t+h/2),
        # k3=F(x+h*k2/2,t+h/2), k4=F(x+h*k3,t+h).
        # F here is probability_flow_drift(model,sde,.,.).
        # Each k has shape [B,2]. Do not change x until all ks are made.
        raise NotImplementedError("TODO 6.2: four RK4 evaluations")

        # TODO 6.3: Weighted average: x += h/6*(k1+2*k2+2*k3+k4).
        # There should be FOUR score-network evaluations per step.
        raise NotImplementedError("TODO 6.3: RK4 state update")

        if record_every and (i + 1) % record_every == 0:
            traj.append(x.cpu().clone())

    return x.cpu(), traj


In [ ]:
# --- SELF-CHECK 5: PF-ODE identity on stationary Gaussian VP marginals --
# For p_t=N(0,I), score=-x and VP f=-beta*x/2, g^2=beta, so
# F = f - g^2*score/2 = 0 exactly! Any RK4 grid should return the input.
initial_gaussian = torch.randn(64, 2, device=device)
t_batch = torch.full((64,), 0.5, device=device)
field = probability_flow_drift(oracle_gaussian, vp, initial_gaussian, 0.5)
assert field.shape == initial_gaussian.shape
assert torch.allclose(field, torch.zeros_like(field), atol=3e-6, rtol=1e-5)

ode_result, ode_traj = pf_ode_sample(
    oracle_gaussian, vp, initial=initial_gaussian, n_steps=16, record_every=4
)
assert len(ode_traj) == 5
assert torch.allclose(ode_result, initial_gaussian.cpu(), atol=3e-5, rtol=1e-5)
print("✓ SELF-CHECK 5 passed: PF-ODE has the correct 1/2 factor and RK4 shape")


In [ ]:
# --- Sample with PF-ODE --------------------------------------------------
samples_ve_ode, _ = pf_ode_sample(
    score_ve, ve,
    n_samples=N_SAMPLES,
    n_steps=ODE_STEPS,
    seed=20,
)

samples_vp_ode, traj_vp_ode = pf_ode_sample(
    score_vp, vp,
    n_samples=N_SAMPLES,
    n_steps=ODE_STEPS,
    seed=21,
    record_every=max(1, ODE_STEPS // 5),
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].scatter(X[:, 0], X[:, 1], s=3, alpha=0.5)
axes[0].set_title("Data")
axes[1].scatter(samples_ve_ode[:, 0], samples_ve_ode[:, 1], s=3, alpha=0.5)
axes[1].set_title("VE PF-ODE (RK4)")
axes[2].scatter(samples_vp_ode[:, 0], samples_vp_ode[:, 1], s=3, alpha=0.5)
axes[2].set_title("VP PF-ODE (RK4)")
for ax in axes:
    ax.axis("equal")
plt.tight_layout()
plt.show()

print("Reverse SDE score evaluations (approx.):", SAMPLE_STEPS)
print("PF-ODE RK4 score evaluations (approx.):", 4 * ODE_STEPS)

# TODO (optional numerical analysis): implement an Euler PF-ODE solver
# with the same signature and compare 20 vs. 75 steps with RK4. Compare
# both steps AND score-network evaluations (RK4 uses 4x per step).


**What to notice:** the SDE and PF-ODE do not trace the same *individual* path. The reverse SDE
keeps injecting calibrated randomness, while the PF-ODE transports points deterministically.
The important Ch. 4 statement is distributional: with the **oracle** score and exact integration,
both reproduce the same family of marginals $p_t$. With a learned score and finite solver steps,
small visual differences are expected because of model error and numerical discretization error.


## 4.6 VP-SDE as Continuous DDPM — A Third Sampler

The learning-path checkpoint asks for the **same trained model** to support three samplers:
ancestral, reverse SDE, and PF-ODE.

Section 4.5 makes this possible without training a separate DDPM noise-predictor. For the affine
kernel

$$
x_s = \alpha_s x_0 + \sigma_s\epsilon,
$$

the learned marginal score implies the posterior-mean clean prediction (generalized Tweedie):

$$
\hat x_0(x_s,s)
=
\frac{x_s+\sigma_s^2 s_\phi(x_s,s)}{\alpha_s}.
$$

For two times $0\le t<s$, Lemma 4.5.2 gives the Gaussian reverse conditional

$$
p(x_t\mid x_s,x_0)
=
\mathcal N\left(x_t;\mu(x_s,x_0;s,t),\sigma^2(s,t)I\right).
$$

Replacing $x_0$ by $\hat x_0$ gives a DDPM-style **ancestral** sampler. This is a discrete
reverse chain; the reverse-SDE sampler is its continuous-time limit under the VP dynamics.


In [ ]:
# --- TODO 7: VP ancestral sampler using the SAME score_vp model ---------
# Ch.4 §4.5 and Lemma 4.5.2: derive Gaussian reverse posterior from the
# affine forward kernel, then plug in x0_hat from the learned score.
# Use n_steps discrete reverse jumps; final jump to t=0 is deterministic.

@torch.no_grad()
def vp_ancestral_sample(
    model,
    sde,
    n_samples=2000,
    dim=2,
    n_steps=500,
    initial=None,
    seed=0,
):
    assert isinstance(sde, VPSDE), "This demo is written for the VP/DDPM case."
    generator = torch.Generator(device=device).manual_seed(seed)
    if initial is None:
        x = sde.prior_sample(n_samples, dim, generator=generator)
    else:
        x = initial.to(device).clone()
        n_samples = x.shape[0]

    # PROVIDED: from s (later, noisier) to t (earlier, cleaner).
    current_times = torch.linspace(sde.T, sde.eps, n_steps, device=device)
    next_times = torch.cat([
        current_times[1:],
        torch.tensor([0.0], device=device),
    ])

    for s_value, t_value in zip(current_times, next_times):
        s = torch.full((n_samples,), float(s_value), device=device)

        # TODO 7.1: Evaluate alpha_s, sigma_s (broadcast [B,1])
        # and score model(x,s). Derive posterior mean x0_hat via Tweedie:
        #       x0_hat = (x + sigma_s^2 * score) / alpha_s
        raise NotImplementedError("TODO 7.1: score-to-clean prediction")

        # PROVIDED: t=0 endpoint uses the denoised estimate directly;
        # there is no artificial Gaussian variance to inject at this step.
        if float(t_value) == 0.0:
            x = x0_hat
            break

        t = torch.full((n_samples,), float(t_value), device=device)

        # TODO 7.2: Evaluate alpha_t, sigma_t and the between-time coeffs:
        # alpha_{s|t} = alpha_s / alpha_t
        # sigma_{s|t}^2 = sigma_s^2 - alpha_{s|t}^2 * sigma_t^2.
        # Clamp the between-time variance to a small positive value.
        raise NotImplementedError("TODO 7.2: between-time forward kernel")

        # TODO 7.3: Gaussian posterior p(x_t | x_s, x0_hat):
        # mean = alpha_{s|t}*(sigma_t^2/sigma_s^2)*x_s
        #      + alpha_t*(sigma_{s|t}^2/sigma_s^2)*x0_hat
        # variance = sigma_{s|t}^2 * sigma_t^2 / sigma_s^2
        # Verify mean/variance can broadcast across x [B,2].
        raise NotImplementedError("TODO 7.3: reverse Gaussian mean and variance")

        # TODO 7.4: sample x_t = mean + sqrt(variance)*epsilon using the
        # supplied torch.Generator. Protect variance from roundoff.
        raise NotImplementedError("TODO 7.4: ancestral Gaussian step")

    return x.cpu()


In [ ]:
# --- SELF-CHECK 6: ancestral solver basic functional contract -----------
# Do not expect sample-by-sample agreement between different samplers.
# This checks successful reverse integration and seeded reproducibility.
fixed_start = torch.randn(80, 2, device=device)
c = vp_ancestral_sample(
    oracle_gaussian, vp, n_steps=32, initial=fixed_start, seed=42
)
d = vp_ancestral_sample(
    oracle_gaussian, vp, n_steps=32, initial=fixed_start, seed=42
)
assert c.shape == (80, 2) and torch.isfinite(c).all()
torch.testing.assert_close(c, d)
assert torch.isfinite(c).all()
print("✓ SELF-CHECK 6 passed: ancestral sampler shape / reproducibility")


In [ ]:
# --- Same VP score model, three samplers --------------------------------
# Use the same starting latent for all three so the difference is sampler-only.
latent_gen = torch.Generator(device=device).manual_seed(123)
shared_latent = vp.prior_sample(N_SAMPLES, 2, generator=latent_gen)

samples_vp_ancestral = vp_ancestral_sample(
    score_vp, vp,
    n_steps=SAMPLE_STEPS,
    initial=shared_latent,
    seed=30,
)

samples_vp_sde_same, _ = reverse_sde_sample(
    score_vp, vp,
    n_steps=SAMPLE_STEPS,
    initial=shared_latent,
    seed=31,
)

samples_vp_ode_same, _ = pf_ode_sample(
    score_vp, vp,
    n_steps=ODE_STEPS,
    initial=shared_latent,
    seed=32,  # irrelevant once `initial` is fixed; ODE itself is deterministic
)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
plots = [
    ("Data", X),
    ("VP ancestral", samples_vp_ancestral),
    ("VP reverse SDE", samples_vp_sde_same),
    ("VP PF-ODE", samples_vp_ode_same),
]
for ax, (title, pts) in zip(axes, plots):
    ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.5)
    ax.set_title(title)
    ax.axis("equal")
plt.tight_layout()
plt.show()

# TODO (independent coding practice): wrap these three sampling calls
# in a helper accepting (model, sde, initial, step_counts); report the
# number of score evaluations for each. Keep the same `shared_latent`.


In [ ]:
# --- Determinism check: SDE changes, PF-ODE does not --------------------
small_latent = shared_latent[:300]

sde_run_1, _ = reverse_sde_sample(
    score_vp, vp,
    n_steps=SAMPLE_STEPS,
    initial=small_latent,
    seed=100,
)
sde_run_2, _ = reverse_sde_sample(
    score_vp, vp,
    n_steps=SAMPLE_STEPS,
    initial=small_latent,
    seed=101,
)

ode_run_1, _ = pf_ode_sample(
    score_vp, vp,
    n_steps=ODE_STEPS,
    initial=small_latent,
)
ode_run_2, _ = pf_ode_sample(
    score_vp, vp,
    n_steps=ODE_STEPS,
    initial=small_latent,
)

print("Mean |SDE run 1 - SDE run 2|:",
      (sde_run_1 - sde_run_2).abs().mean().item())
print("Max  |ODE run 1 - ODE run 2|:",
      (ode_run_1 - ode_run_2).abs().max().item())

# TODO (optional experiment): repeat the SDE run for five seeds and
# summarize the spread in output means, while the ODE results remain
# bitwise identical when the initial latent is held fixed.


## 4.7 Why the Marginals Match: Fokker–Planck in One Line

For a scalar-diffusion forward SDE

$$
dx = f(x,t)\,dt + g(t)\,dw,
$$

the marginal density satisfies the **Fokker–Planck equation**

$$
\partial_t p_t(x)
=
-\nabla\cdot\big(f(x,t)p_t(x)\big)
+
\frac12 g(t)^2\Delta p_t(x).
$$

The reverse-time SDE and PF-ODE are engineered so their density evolution reproduces this same
$p_t$ family. That is the unifying principle behind the apparent coincidence:

- VE-SDE and VP-SDE can use the same training recipe.
- The reverse SDE and PF-ODE can use the same learned score.
- DDPM ancestral sampling appears as a discretized VP reverse process.
- NCSN appears as a discretized VE process.

So the conceptual jump in Ch. 4 is not "a new diffusion model." It is a **continuous-time language**
in which the models from Ch. 2 and Ch. 3 become special cases and sampling becomes a numerical
differential-equation problem.


## 8. Exercises / Checkpoint

1. **VE vs. VP forward process.** Change `sigma_max` in `VESDE` and `beta_max` in `VPSDE`.
   Which parameter has to be large enough for $p_T$ to become essentially independent of $x_0$?
2. **Discretization error.** Reduce `SAMPLE_STEPS` from 300 to 50. Which degrades faster:
   the ancestral sampler or Euler–Maruyama reverse SDE?
3. **ODE order.** Replace RK4 with a one-line Euler PF-ODE update. At the same number of steps,
   which solver preserves the two-moons geometry better?
4. **Stochasticity.** Reuse one fixed `shared_latent` and run the reverse SDE with five different
   random seeds. Then do the same with PF-ODE. Explain the difference.
5. **Score ↔ noise prediction.** For the VP model, verify numerically that
   $\hat\epsilon(x_t,t)=-\sigma_t s_\phi(x_t,t)$ and use it to rewrite the ancestral sampler in
   DDPM-style $\epsilon$-prediction notation.
6. **Optional theory.** Starting from the VP perturbation kernel, differentiate
   $\alpha_t=\exp[-\frac12\int_0^t\beta(s)ds]$ and recover
   $f(x,t)=-\frac12\beta(t)x$.

**Checkpoint:** with the **same trained VP score network**, the ancestral sampler, reverse-time SDE,
and PF-ODE should all turn a Gaussian prior into visually consistent two-moons samples. You should
also be able to explain why the SDE is stochastic, why the PF-ODE is deterministic, and why both
can still agree at the level of marginal distributions.
